# AusGrid2013: DP-FedAvg training with client sampling

The defaults are a sampling rate of `0.1`, 100 rounds, and deduplicated data. Edit the parameters, then select Run All.
Each round uniformly samples `ceil(q × N)` clients without replacement, with at least one client.
For q<1, RDP accounting automatically matches fixed-size sampling without replacement. SAMPLING_SEED=None uses fresh, hidden randomness for both sampling and noise.
For add/remove adjacency with partial participation, noise is calibrated to 2C/m. Only the global models and privacy reports in model_release/ are covered by the accountant.
Outputs are written separately to `DP_fhf/outputs/sampling_experiments/`, including forecasts, training residuals, privacy metadata, and per-round participation logs.


In [ ]:
# Locate the project package from a notebook folder or the project root.
from pathlib import Path
import os
import sys
_pphf_start = Path.cwd().resolve()
_pphf_override = os.environ.get("PPHF_PROJECT_ROOT") or os.environ.get("DP_FHF_PROJECT_ROOT")
_pphf_candidates = ([Path(_pphf_override).expanduser()] if _pphf_override else []) + [_pphf_start, *_pphf_start.parents]
PROJECT_ROOT = next((p.resolve() for p in _pphf_candidates if (p / "PPHF_modules" / "fhf" / "FHF_runner.py").is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Cannot locate PPHF_modules. Set PPHF_PROJECT_ROOT or start Jupyter inside the project.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from pathlib import Path
import os
import sys

PROJECT_ROOT = os.environ.get("DP_FHF_PROJECT_ROOT")

def locate_project_root():
    candidates = [Path(PROJECT_ROOT).expanduser()] if PROJECT_ROOT else []
    cwd = Path.cwd().resolve()
    candidates.extend([cwd, *cwd.parents])
    for candidate in candidates:
        if (candidate / "PPHF_modules" / "fhf").is_dir() and (candidate / "DP_fhf").is_dir():
            return candidate.resolve()
    raise FileNotFoundError(
        "Cannot locate the FHF project root. Set DP_FHF_PROJECT_ROOT or PROJECT_ROOT."
    )

ROOT = locate_project_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from PPHF_modules.dp_fhf.notebook_runtime import (
    locate_dataset_csv,
    run_covariance_notebook,
    run_fixed_dpfedavg,
    run_o2mr_notebook,
    run_sweep_dpfedavg,
    summarize_dpfedavg,
)

DRY_RUN = os.environ.get("DP_FHF_NOTEBOOK_DRY_RUN", "0") == "1"
print("Project root:", ROOT)
print("Python:", sys.executable)
print("Dry run:", DRY_RUN)

In [ ]:
DATASET = "AusGrid2013"
SAMPLING_RATE = 0.5
SAMPLING_SEED = None  # Fresh hidden randomness for partial participation
DATA_CSV_OVERRIDE = None  # Optional explicit CSV path
CSV_PATH = locate_dataset_csv(ROOT, DATASET, DATA_CSV_OVERRIDE)
print("Deduplicated input:", CSV_PATH)
NUM_ROUNDS = 100
OVERWRITE = False


In [ ]:
result = run_fixed_dpfedavg(
    root=ROOT,
    dataset=DATASET,
    csv_path=CSV_PATH,
    num_rounds=NUM_ROUNDS,
    clip_norm=1.0,
    noise_multiplier=1.2,
    sampling_rate=SAMPLING_RATE,
    sampling_seed=SAMPLING_SEED,
    dry_run=DRY_RUN,
    overwrite=OVERWRITE,
    output_base_dir=ROOT / 'DP_fhf' / 'outputs' / 'sampling_experiments' / DATASET,
)
result